In [ ]:
from file_aggregation import file_aggregation
from cnn_model import ECGCNN1D

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

X, y, groups = file_aggregation('/Users/reymendoza/Downloads/mit-bih-arrhythmia-database-1.0.0')

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

gkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
split = gkf.split(X.numpy(), y.squeeze(1).numpy(), groups=groups)

fold_results = []

for fold, (train_idx, test_idx) in enumerate(split, start=1):

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    print(f"Fold {fold}")

    train_ds = TensorDataset(X_train, y_train)
    test_ds = TensorDataset(X_test, y_test)

    y_train_flat = y_train.squeeze(1).long()
    n_pos = (y_train_flat == 1).sum().item()
    n_neg = (y_train_flat == 0).sum().item()

    class_weights = torch.tensor([1.0 / max(n_neg, 1), 1.0 / max(n_pos, 1)], dtype=torch.float32)
    sample_weights = class_weights[y_train_flat]

    sampler = WeightedRandomSampler(
        weights=sample_weights, 
        num_samples=len(sample_weights), 
        replacement=True
    )

    train_loader = DataLoader(train_ds, batch_size=128, sampler=sampler)
    test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

    model = ECGCNN1D()

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)

    epochs = 5

    for epoch in range(epochs):

        model.train()

        running_loss = 0.0

        for xb, yb in train_loader:

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * xb.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)

        model.eval()

        all_probs = []
        all_preds = []
        all_true = []

        with torch.no_grad():
            for xb, yb in test_loader:

                logits = model(xb)
                probs = torch.sigmoid(logits)
                preds = (probs >= 0.5).float()

                all_probs.append(probs)
                all_preds.append(preds)
                all_true.append(yb)

        y_prob = torch.cat(all_probs).numpy().ravel()
        y_pred = torch.cat(all_preds).numpy().ravel()
        y_true = torch.cat(all_true).numpy().ravel()

        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        auc = roc_auc_score(y_true, y_prob)
        cm = confusion_matrix(y_true, y_pred)

        print(
        f"Epoch {epoch+1:02d} | "
        f"Loss: {epoch_loss:.4f} | "
        f"Acc: {acc:.4f} | "
        f"Prec: {prec:.4f} | "
        f"Rec: {rec:.4f} | "
        f"F1: {f1:.4f} | "
        f"AUC: {auc:.4f} "
    )

    fold_results.append({
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "auc": auc, 
        "cm": cm
    })

print(fold_results)

Found 46 valid records.
Excluded records: ['102', '104']
Creating RawArray with float64 data, n_channels=1, n_times=180600
    Range : 0 ... 180599 =      0.000 ...  1805.990 secs
Ready.
Creating RawArray with float64 data, n_channels=1, n_times=180600
    Range : 0 ... 180599 =      0.000 ...  1805.990 secs
Ready.
Creating RawArray with float64 data, n_channels=1, n_times=180600
    Range : 0 ... 180599 =      0.000 ...  1805.990 secs
Ready.
Creating RawArray with float64 data, n_channels=1, n_times=180600
    Range : 0 ... 180599 =      0.000 ...  1805.990 secs
Ready.
Creating RawArray with float64 data, n_channels=1, n_times=180600
    Range : 0 ... 180599 =      0.000 ...  1805.990 secs
Ready.
Creating RawArray with float64 data, n_channels=1, n_times=180600
    Range : 0 ... 180599 =      0.000 ...  1805.990 secs
Ready.
Creating RawArray with float64 data, n_channels=1, n_times=180600
    Range : 0 ... 180599 =      0.000 ...  1805.990 secs
Ready.
Creating RawArray with float64 da

In [5]:
cm

array([[13264,  3430],
       [ 3543,  1282]])

In [6]:
fold_results

[{'accuracy': 0.6616525860292602,
  'precision': 0.27550357374918777,
  'recall': 0.16351716158889318,
  'f1': 0.20522749273959343,
  'auc': 0.3731675621606994,
  'cm': array([[11996,  2230],
         [ 4338,   848]])},
 {'accuracy': 0.4144681300126636,
  'precision': 0.28744693753790174,
  'recall': 0.8909774436090225,
  'f1': 0.4346629986244842,
  'auc': 0.6580281984968067,
  'cm': array([[ 3589, 10575],
         [  522,  4266]])},
 {'accuracy': 0.47573872472783824,
  'precision': 0.268707100258805,
  'recall': 0.39786737754081974,
  'f1': 0.3207737255692122,
  'auc': 0.49270155357606066,
  'cm': array([[6789, 6499],
         [3614, 2388]])},
 {'accuracy': 0.7000045460744647,
  'precision': 0.44985932475884244,
  'recall': 0.7993215497232637,
  'f1': 0.5757088664566321,
  'auc': 0.767363341131317,
  'cm': array([[10921,  5475],
         [ 1124,  4477]])},
 {'accuracy': 0.6759607788466007,
  'precision': 0.2720713073005093,
  'recall': 0.26569948186528497,
  'f1': 0.2688476460102758,
